In [9]:
%pip -q install requests pandas numpy ipython

Note: you may need to restart the kernel to use updated packages.


In [10]:
import io
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from IPython.display import display

STATION_ID = "51201"
LATITUDE, LONGITUDE = 21.671, -158.118
NOAA_URL = f"https://www.ndbc.noaa.gov/data/realtime2/{STATION_ID}.txt"
OPEN_METEO_URL = "https://marine-api.open-meteo.com/v1/marine"
CDIP_URL = "https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/realtime/106p1_rt.nc"

OUTPUT_DIR = Path("waimea_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "WaimeaWaveCast-Academic-Notebook/1.0"})
print("Thời điểm chạy (UTC):", datetime.now(timezone.utc).isoformat())

Thời điểm chạy (UTC): 2026-11-01T08:55:07.228471+00:00


In [11]:
def fetch_noaa_latest() -> pd.DataFrame:
    response = SESSION.get(NOAA_URL, timeout=30)
    response.raise_for_status()
    lines = response.text.splitlines()
    if len(lines) < 3 or not lines[0].startswith("#"):
        raise ValueError("NOAA trả về dữ liệu không đúng định dạng mong đợi")

    columns = lines[0].lstrip("#").split()
    columns[:5] = ["year", "month", "day", "hour", "minute"]
    body = "\n".join(line for line in lines[2:] if line.strip() and not line.startswith("#"))
    df = pd.read_csv(io.StringIO(body), sep=r"\s+", names=columns, na_values=["MM", "mm"])
    if df.empty:
        raise ValueError("NOAA không có bản ghi nào")

    for col in columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["timestamp_utc"] = pd.to_datetime(
        {"year": df["year"], "month": df["month"], "day": df["day"],
         "hour": df["hour"], "minute": df["minute"]},
        utc=True, errors="coerce"
    )
    df = df.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc")
    df = df.drop_duplicates(subset=["timestamp_utc"], keep="last").reset_index(drop=True)
    df["source"] = "NOAA_51201"
    df["data_type"] = "observation"
    return df

noaa_df = fetch_noaa_latest()
show_cols = [c for c in ["timestamp_utc", "WVHT", "DPD", "APD", "MWD", "WTMP", "source"] if c in noaa_df]
print(f"Số bản ghi NOAA nhận được: {len(noaa_df):,}")
print("Bản ghi MỚI NHẤT NOAA:")
display(noaa_df[show_cols].tail(1))
print("5 bản ghi gần nhất:")
display(noaa_df[show_cols].tail(5).sort_values("timestamp_utc", ascending=False))

Số bản ghi NOAA nhận được: 2,157
Bản ghi MỚI NHẤT NOAA:


,timestamp_utc,WVHT,DPD,APD,MWD,WTMP,source
2156,2026-11-01 08:26:00+00:00,1.7,20,9.4,311,26.2,NOAA_51201


5 bản ghi gần nhất:


,timestamp_utc,WVHT,DPD,APD,MWD,WTMP,source
2156,2026-11-01 08:26:00+00:00,1.7,20,9.4,311,26.2,NOAA_51201
2155,2026-11-01 07:56:00+00:00,1.6,20,8.4,313,26.2,NOAA_51201
2154,2026-11-01 07:26:00+00:00,1.6,20,7.7,307,26.2,NOAA_51201
2153,2026-11-01 06:56:00+00:00,1.7,20,7.8,302,26.2,NOAA_51201
2152,2026-11-01 06:26:00+00:00,1.7,20,8.5,314,26.2,NOAA_51201


In [12]:
latest_noaa = noaa_df["timestamp_utc"].max()
now_utc = pd.Timestamp.now(tz="UTC")
age = now_utc - latest_noaa
print("Mốc quan trắc NOAA mới nhất (UTC):", latest_noaa)
print("Giờ Hawaii:", latest_noaa.tz_convert("Pacific/Honolulu"))
print("Độ trễ dữ liệu (phút):", round(age.total_seconds()/60, 1))
if age > pd.Timedelta(hours=2):
    print("CẢNH BÁO: Quan trắc NOAA đã cũ trên 2 giờ; kiểm tra CDIP hoặc dữ liệu mô hình.")
else:
    print("Quan trắc NOAA nằm trong ngưỡng mới gần đây (<= 2 giờ, ngưỡng kiểm thử).")

noaa_path = OUTPUT_DIR / "noaa_51201_latest_window.csv"
noaa_df.to_csv(noaa_path, index=False)
print("Đã xuất:", noaa_path.resolve())

Mốc quan trắc NOAA mới nhất (UTC): 2026-11-01 08:26:00+00:00
Giờ Hawaii: 2026-10-31 22:26:00-10:00
Độ trễ dữ liệu (phút): 29.1
Quan trắc NOAA nằm trong ngưỡng mới gần đây (<= 2 giờ, ngưỡng kiểm thử).
Đã xuất: E:\23115053122114_K23_PhanVanKhanh\College_Year04\126\Kho-du-lieu\repo\waimea_data\noaa_51201_latest_window.csv


In [13]:
OPEN_METEO_PARAMS = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": "wave_height,wave_period,wave_direction,swell_wave_height,sea_surface_temperature",
    "past_days": 1,
    "forecast_days": 2,
    "timezone": "UTC",
}
response = SESSION.get(OPEN_METEO_URL, params=OPEN_METEO_PARAMS, timeout=30)
print("Request URL:", response.url)
response.raise_for_status()
meteo_json = response.json()
if "hourly" not in meteo_json:
    raise ValueError(f"Open-Meteo không trả về hourly: {str(meteo_json)[:400]}")

meteo_df = pd.DataFrame(meteo_json["hourly"])
meteo_df["timestamp_utc"] = pd.to_datetime(meteo_df.pop("time"), utc=True)
meteo_df["source"] = "OPEN_METEO_MARINE"
meteo_df["data_type"] = "model_forecast"
meteo_df = meteo_df.sort_values("timestamp_utc")
now_utc = pd.Timestamp.now(tz="UTC")
current = meteo_df.loc[meteo_df["timestamp_utc"] <= now_utc]
current = current.dropna(subset=["wave_height"])
future = meteo_df.loc[meteo_df["timestamp_utc"] > now_utc]

if current.empty:
    print("Chưa có giá trị mô hình hợp lệ trước thời điểm hiện tại.")
else:
    print("Giá trị MÔ HÌNH gần nhất (KHÔNG PHẢI số đo phao):")
    display(current.tail(1))
print("6 mốc dự báo tiếp theo:")
display(future.head(6))

meteo_path = OUTPUT_DIR / "open_meteo_waimea_hourly.csv"
meteo_df.to_csv(meteo_path, index=False)
print("Đã xuất:", meteo_path.resolve())

Request URL: https://marine-api.open-meteo.com/v1/marine?latitude=21.671&longitude=-158.118&hourly=wave_height%2Cwave_period%2Cwave_direction%2Cswell_wave_height%2Csea_surface_temperature&past_days=1&forecast_days=2&timezone=UTC
Giá trị MÔ HÌNH gần nhất (KHÔNG PHẢI số đo phao):


,wave_height,wave_period,wave_direction,swell_wave_height,sea_surface_temperature,timestamp_utc,source,data_type
32,1.66,13.7,349,1.06,26.5,2026-11-01 08:00:00+00:00,OPEN_METEO_MARINE,model_forecast


6 mốc dự báo tiếp theo:


,wave_height,wave_period,wave_direction,swell_wave_height,sea_surface_temperature,timestamp_utc,source,data_type
33,1.70,14.00,343,1.08,26.5,2026-11-01 09:00:00+00:00,OPEN_METEO_MARINE,model_forecast
34,1.74,14.25,341,1.14,26.5,2026-11-01 10:00:00+00:00,OPEN_METEO_MARINE,model_forecast
35,1.76,14.45,338,1.22,26.5,2026-11-01 11:00:00+00:00,OPEN_METEO_MARINE,model_forecast
36,1.80,14.65,336,1.28,26.5,2026-11-01 12:00:00+00:00,OPEN_METEO_MARINE,model_forecast
37,1.84,14.80,335,1.34,26.5,2026-11-01 13:00:00+00:00,OPEN_METEO_MARINE,model_forecast
38,1.86,15.00,334,1.40,26.5,2026-11-01 14:00:00+00:00,OPEN_METEO_MARINE,model_forecast


Đã xuất: E:\23115053122114_K23_PhanVanKhanh\College_Year04\126\Kho-du-lieu\repo\waimea_data\open_meteo_waimea_hourly.csv


In [14]:
%pip -q install xarray netCDF4

Note: you may need to restart the kernel to use updated packages.


In [15]:
import numpy as np
import xarray as xr

try:
    ds = xr.open_dataset(CDIP_URL, engine="netcdf4", decode_times=False, mask_and_scale=True)
    try:
        print("CDIP dataset:", CDIP_URL)
        tail = slice(-24, None)
        col_map = {
            "waveTime": "timestamp_unix",
            "waveHs": "wave_height_m",
            "waveTp": "peak_period_s",
            "waveTa": "average_period_s",
            "waveDp": "wave_direction_deg",
            "waveFlagPrimary": "qc_flag",
        }
        values = {}
        for src, dst in col_map.items():
            if src in ds:
                values[dst] = np.asarray(ds[src].isel(waveTime=tail).values).ravel()
        cdip_df = pd.DataFrame(values)
        if "timestamp_unix" not in cdip_df:
            raise ValueError("Không có trường waveTime trong CDIP dataset")
        cdip_df["timestamp_utc"] = pd.to_datetime(cdip_df.pop("timestamp_unix"), unit="s", utc=True)
        cdip_df = cdip_df.sort_values("timestamp_utc")
        cdip_df["source"] = "CDIP_106"
        cdip_df["data_type"] = "observation"
        print("Các bản ghi gần nhất CDIP (QC flag xem trước khi dùng):")
        display(cdip_df.tail(5).sort_values("timestamp_utc", ascending=False))
        cdip_df.to_csv(OUTPUT_DIR / "cdip_106_latest_24.csv", index=False)
    finally:
        ds.close()
except Exception as exc:
    print("Không đọc được CDIP trong lần thử này:", type(exc).__name__, str(exc)[:350])
    print("NOAA/Open-Meteo vẫn chạy độc lập. Kiểm tra URL THREDDS, kết nối mạng và netCDF4.")

Không đọc được CDIP trong lần thử này: OSError [Errno -78] NetCDF: Authorization failure: 'https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/realtime/106p1_rt.nc'
NOAA/Open-Meteo vẫn chạy độc lập. Kiểm tra URL THREDDS, kết nối mạng và netCDF4.


In [16]:

import requests

url = "https://cdip.ucsd.edu/data_access/ndar.cdip?106+pm+1+h"

try:
    response = requests.get(url, timeout=30)
    print("HTTP Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))

    response.raise_for_status()

    print("\nDữ liệu CDIP trả về:")
    print(response.text[:3000])

except requests.RequestException as exc:
    print("Lỗi truy cập CDIP:", exc)
#done check - ThoKhang


HTTP Status: 403
Content-Type: text/html
Lỗi truy cập CDIP: 403 Client Error: Forbidden for url: https://cdip.ucsd.edu/data_access/ndar.cdip?106+pm+1+h
